# Water-Level Forecast & Flood Warning — t+1

**Objective**  
Single end-to-end pipeline that:
1. Forecasts **next-day peak water level** (t+1).
2. Issues a **binary flood warning** (level ≥ I).
3. Classifies severity into **4 administrative levels**: 0 / I / II / III  
   (original level IV is merged into III because of extremely low sample size).
4. Exports a ready-to-deploy package.

**Data split (time-based, no leakage)**  
| Split       | Period          | Purpose                          |
|-------------|-----------------|----------------------------------|
| Train       | year ≤ 2019     | Model fitting                    |
| Validation  | year = 2020     | Early stopping + conformal q90   |
| Test        | year ≥ 2021     | Final evaluation                 |

**Model**  
Equal-weight ensemble: **LightGBM + XGBoost + CatBoost**.

**Target**  
`Δ = future_peak − current_water_level`  
Final prediction: `pred_peak = y_t + predicted_Δ`

**Binary flood definition**  
Any day whose peak reaches **administrative level ≥ I** (≥ 1.40 m) is treated as a positive flood event.

**Severity levels (merged)**  
| Level | Water level (m)      |
|-------|----------------------|
| 0     | < 1.40               |
| I     | 1.40 – < 1.50        |
| II    | 1.50 – < 1.60        |
| III   | ≥ 1.60 (includes old IV) |

## 0. Setup & Configuration

In [ ]:
!pip install pandas numpy scikit-learn lightgbm xgboost catboost joblib openpyxl

In [ ]:
import json
import shutil
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    precision_recall_fscore_support, classification_report
)
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore")

try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except Exception as e:
    HAS_CAT = False
    raise ImportError("CatBoost is required for the final ensemble.") from e

# Resolve the repository root from the notebook location or current working directory.
NOTEBOOK_PATH = Path.cwd()
SEARCH_ROOTS = [NOTEBOOK_PATH, *NOTEBOOK_PATH.parents]
ROOT = next((p for p in SEARCH_ROOTS if (p / "data").is_dir()), NOTEBOOK_PATH)
DATA_DIR = ROOT / "data"
OUT = ROOT / "artifacts"
OUT.mkdir(parents=True, exist_ok=True)

MODEL_DIR = OUT / "water_flood_t1"
if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS = [1]
THR_I, THR_II, THR_III = 1.40, 1.50, 1.60
BETA = 2.0

STATION_MAP = {
    "LeMinhXuan": "Lê Minh Xuân",
    "HocMon": "Hóc Môn",
    "NhaBe": "Nhà Bè",
    "PhuAn": "Phú An",
    "ThuDuc": "Thủ Đức",
}

REQUIRED = [
    "stations_master_features.csv",
    "feature_matrix_external_clean.csv",
    "tide_hourly.csv",
    "rain_daily_station_obs.csv",
    "historical_weather_features_openmeteo_bangkok.csv",
]


def find_file(name: str) -> Path | None:
    candidates = [DATA_DIR / name, ROOT / name, Path(name)]
    return next((path for path in candidates if path.exists()), None)


def must_find(name: str) -> Path:
    path = find_file(name)
    if path is None:
        raise FileNotFoundError(f"Missing project data file: {name}")
    return path


def rmse(y, p) -> float:
    return float(np.sqrt(mean_squared_error(y, p)))


def mae(y, p) -> float:
    return float(mean_absolute_error(y, p))


def r2(y, p) -> float:
    return float(r2_score(y, p))


def binary_metrics(y_true, y_pred, beta: float = 2.0) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    b2 = beta ** 2
    fb = (1 + b2) * p * r / (b2 * p + r + 1e-12)
    return {"precision": float(p), "recall": float(r), "f1": float(f1), "fbeta": float(fb), "support": int(y_true.sum()), "pred_pos": int(y_pred.sum())}


def peak_to_alarm(peak: float) -> int:
    if peak < THR_I:
        return 0
    if peak < THR_II:
        return 1
    if peak < THR_III:
        return 2
    return 3


def best_fbeta_threshold(y_true, y_score, beta: float = 2.0):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=float)
    if y_true.sum() == 0 or len(np.unique(y_score)) == 1:
        return float(THR_I), 0.0
    order = np.argsort(-y_score)
    ys, sc = y_true[order], y_score[order]
    tp = np.cumsum(ys)
    fp = np.cumsum(1 - ys)
    fn = tp[-1] - tp
    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    b2 = beta ** 2
    fb = (1 + b2) * precision * recall / (b2 * precision + recall + 1e-12)
    i = int(np.argmax(fb))
    return float(sc[i]), float(fb[i])


print(f"Project root: {ROOT}")
print(f"Data dir    : {DATA_DIR}")
print(f"Output dir  : {OUT}")

LightGBM  : 4.6.0
XGBoost   : 3.2.0
CatBoost  : True
Output dir: /kaggle/working


## 1. Verify input files

In [2]:
for name in REQUIRED:
    p = find_file(name)
    status = "OK" if p else "MISSING"
    print(f"{status:7s} {name}")

assert all(find_file(n) for n in REQUIRED), "One or more required files are missing."

OK      stations_master_features.csv
OK      feature_matrix_external_clean.csv
OK      tide_nhabe_clean.csv
OK      rain_daily_station_obs.csv
OK      historical_weather_features.csv
OK      flood_binary_2022_5_stations_long.csv


## 2. Tidal, lunar & upstream-rain features

Construct daily tidal statistics, lags, lunar / spring-neap cycle features,  
and precipitation / weather variables at the upstream reservoirs (Trị An, Dầu Tiếng).

> Note: Trị An / Dầu Tiếng values represent **upstream rainfall**, not actual reservoir discharge.

In [ ]:
def add_lunar_features(df: pd.DataFrame, date_col: str = "ngay") -> pd.DataFrame:
    ref = pd.Timestamp("2000-01-06")
    synodic = 29.530588
    days = (df[date_col] - ref).dt.total_seconds() / 86400.0
    lunar_age = np.mod(days, synodic)

    df = df.copy()
    df["lunar_day"] = lunar_age
    df["moon_phase_sin"] = np.sin(2 * np.pi * lunar_age / synodic)
    df["moon_phase_cos"] = np.cos(2 * np.pi * lunar_age / synodic)
    df["spring_neap_sin"] = np.sin(2 * np.pi * lunar_age / (synodic / 2.0))
    df["spring_neap_cos"] = np.cos(2 * np.pi * lunar_age / (synodic / 2.0))
    return df


# ---- Tide ----
raw = pd.read_csv(must_find("tide_hourly.csv"))
raw["datetime_vungtau"] = pd.to_datetime(raw["datetime_nhabe"], errors="raise")
raw = raw.dropna(subset=["datetime_vungtau", "tide_vungtau_m"]).sort_values("datetime_vungtau")
raw["ngay"] = raw["datetime_vungtau"].dt.floor("D")

tide = (
    raw.groupby("ngay", as_index=False)
    .agg(
        tide_max=("tide_vungtau_m", "max"),
        tide_min=("tide_vungtau_m", "min"),
        tide_mean=("tide_vungtau_m", "mean"),
    )
)
tide["tide_range"] = tide["tide_max"] - tide["tide_min"]
tide = tide.sort_values("ngay").reset_index(drop=True)

for lag in [1, 2, 3, 5, 7]:
    for c in ["tide_max", "tide_mean", "tide_range"]:
        tide[f"{c}_lag{lag}"] = tide[c].shift(lag)

tide["doy"] = tide["ngay"].dt.dayofyear
tide["tide_doy_sin"] = np.sin(2 * np.pi * tide["doy"] / 365.25)
tide["tide_doy_cos"] = np.cos(2 * np.pi * tide["doy"] / 365.25)
tide = add_lunar_features(tide, "ngay")

# ---- Weather / upstream rain ----
wx = pd.read_csv(must_find("historical_weather_features_openmeteo_bangkok.csv"))
wx["date"] = pd.to_datetime(wx["date"], errors="raise")
wx["ngay"] = wx["date"].dt.floor("D")

SITES = [
    "Dap_Tri_An", "Dap_Dau_Tieng",
    "Tram_Nha_Be", "Tram_Phu_An", "Tram_Hoc_Mon", "Tram_Thu_Duc",
]
VARS = ["precipitation", "rain", "wind_speed_10m", "wind_gusts_10m", "pressure_msl"]

agg = {}
for s in SITES:
    for v in VARS:
        col = f"{s}_{v}"
        if col in wx.columns:
            agg[col] = "sum" if v in ("precipitation", "rain") else "mean"

wx_daily = wx.groupby("ngay", as_index=False).agg(agg)
wx_daily = wx_daily.rename(
    columns={
        c: c.replace("Dap_", "dam_").replace("Tram_", "st_")
        for c in wx_daily.columns if c != "ngay"
    }
).sort_values("ngay")

for dam in ["dam_Tri_An", "dam_Dau_Tieng"]:
    for metric in ["precipitation", "rain"]:
        col = f"{dam}_{metric}"
        if col in wx_daily.columns:
            wx_daily[f"{col}_lag1"] = wx_daily[col].shift(1)
            wx_daily[f"{col}_sum3"] = wx_daily[col].rolling(3, min_periods=1).sum()

for lag in [1, 2, 3]:
    for c in list(wx_daily.columns):
        if c == "ngay":
            continue
        if any(k in c for k in ["precipitation", "rain", "wind_gusts", "pressure_msl"]) and "_lag" not in c and "_sum" not in c:
            wx_daily[f"{c}_lag{lag}"] = wx_daily[c].shift(lag)

print(f"Tide shape   : {tide.shape}")
print(f"Weather shape: {wx_daily.shape}")

Tide shape   : (6808, 28)
Weather shape: (2557, 107)


## 3. Master dataset & targets

In [4]:
sta = pd.read_csv(must_find("stations_master_features.csv"))
sta_hi = sta[sta["tide_inf_sigmoid"] > 0.9].copy()
sta_hi = sta_hi[~sta_hi["tenTram"].isin(["Củ Chi", "Cần Giờ"])]
stations = sta_hi["tenTram"].tolist()
print("Stations:", stations)

fm = pd.read_csv(must_find("feature_matrix_external_clean.csv"))
fm["ngay"] = pd.to_datetime(fm["ngay"])
fm = fm[fm["tenTram"].isin(stations)].copy()

base_cols = [
    "ngay", "tenTram", "doCaoDinhT", "alarm_level",
    "baoDongI", "baoDongII", "baoDongIII",
    "rain", "rain_max_intensity", "rain_hours",
    "rain_lag1", "rain_lag2", "rain_lag3", "rain_lag7",
    "rain_roll3", "rain_roll7",
    "month", "day", "month_sin", "month_cos", "doy_sin", "doy_cos",
    "dist_to_river_m", "tide_inf_sigmoid", "tide_inf_exp", "pct_impervious",
    "channel_length_m", "channel_density_m_m2", "lon", "lat",
]
keep = [c for c in base_cols if c in fm.columns]

master = (
    fm[keep]
    .merge(tide, on="ngay", how="left")
    .merge(wx_daily, on="ngay", how="left")
    .sort_values(["tenTram", "ngay"])
    .reset_index(drop=True)
)
master["rain"] = master.get("rain", pd.Series(0, index=master.index)).fillna(0)

for h in HORIZONS:
    master[f"target_h{h}"] = master.groupby("tenTram")["doCaoDinhT"].shift(-h)
    master[f"delta_h{h}"] = master[f"target_h{h}"] - master["doCaoDinhT"]

master["year"] = master["ngay"].dt.year
master.to_csv(OUT / "master_t1.csv", index=False)
print(f"Master shape: {master.shape}")

Stations: ['Hóc Môn', 'Lê Minh Xuân', 'Thủ Đức', 'Nhà Bè', 'Phú An']
Master shape: (10721, 166)


## 4. Feature schema

`y_t` (current water level) is retained as a critical feature for delta forecasting.

In [5]:
ban = {
    "ngay", "tenTram", "year", "doCaoDinhT",
    "alarm_level", "baoDongI", "baoDongII", "baoDongIII",
}
ban |= {f"target_h{h}" for h in HORIZONS} | {f"delta_h{h}" for h in HORIZONS}

FEAT = [c for c in master.columns if c not in ban and master[c].dtype != "O"]
if "station_mean_peak" not in FEAT:
    FEAT.append("station_mean_peak")
if "y_t" not in FEAT:
    FEAT.append("y_t")

print(f"Feature count: {len(FEAT)}")
print(FEAT)

Feature count: 158
['rain', 'rain_max_intensity', 'rain_hours', 'rain_lag1', 'rain_lag2', 'rain_lag3', 'rain_lag7', 'rain_roll3', 'rain_roll7', 'month', 'day', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'dist_to_river_m', 'tide_inf_sigmoid', 'tide_inf_exp', 'pct_impervious', 'channel_length_m', 'channel_density_m_m2', 'lon', 'lat', 'tide_max', 'tide_min', 'tide_mean', 'tide_range', 'tide_max_lag1', 'tide_mean_lag1', 'tide_range_lag1', 'tide_max_lag2', 'tide_mean_lag2', 'tide_range_lag2', 'tide_max_lag3', 'tide_mean_lag3', 'tide_range_lag3', 'tide_max_lag5', 'tide_mean_lag5', 'tide_range_lag5', 'tide_max_lag7', 'tide_mean_lag7', 'tide_range_lag7', 'doy', 'tide_doy_sin', 'tide_doy_cos', 'lunar_day', 'moon_phase_sin', 'moon_phase_cos', 'spring_neap_sin', 'spring_neap_cos', 'dam_Tri_An_precipitation', 'dam_Tri_An_rain', 'dam_Tri_An_wind_speed_10m', 'dam_Tri_An_wind_gusts_10m', 'dam_Tri_An_pressure_msl', 'dam_Dau_Tieng_precipitation', 'dam_Dau_Tieng_rain', 'dam_Dau_Tieng_wind_speed_10m

## 5. Ensemble training

- **Train (≤ 2019)**: model fitting  
- **Validation (2020)**: early stopping + conformal calibration  
- **Test (≥ 2021)**: final unbiased evaluation  

Ensemble = equal-weight average of LightGBM + XGBoost + CatBoost.

In [6]:
def make_weights(y_abs, thr=THR_I, alpha=3.0, cap=6.0):
    y = np.asarray(y_abs, dtype=float)
    excess = np.clip(y - thr, 0, None)
    return np.exp(np.clip(alpha * excess, 0, np.log(cap)))


def fit_blend(Xtr, ytr, sw, Xval, yval):
    models = {}

    ml = lgb.LGBMRegressor(
        n_estimators=600, learning_rate=0.04, max_depth=6, num_leaves=48,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbosity=-1,
    )
    ml.fit(
        Xtr, ytr, sample_weight=sw,
        eval_set=[(Xval, yval)],
        callbacks=[lgb.early_stopping(40, verbose=False)],
    )
    models["lgb"] = ml

    mx = xgb.XGBRegressor(
        n_estimators=600, learning_rate=0.04, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, early_stopping_rounds=40, n_jobs=-1,
        objective="reg:squarederror",
    )
    mx.fit(Xtr, ytr, sample_weight=sw, eval_set=[(Xval, yval)], verbose=False)
    models["xgb"] = mx

    if HAS_CAT:
        mc = CatBoostRegressor(
            iterations=600, learning_rate=0.04, depth=6,
            verbose=False, random_seed=42,
        )
        mc.fit(
            Xtr, ytr, sample_weight=sw,
            eval_set=(Xval, yval),
            early_stopping_rounds=40,
        )
        models["cat"] = mc

    return models


def predict_blend(models, X):
    return np.mean([m.predict(X) for m in models.values()], axis=0)


models_h = {}
preds_validation = {}
preds_test = {}
metrics_peak = []
station_maps = {}

for h in HORIZONS:
    print("\n" + "=" * 70)
    print(f"TRAIN / VALIDATION / TEST — t+{h}")

    tr = master[(master.year <= 2019)].dropna(subset=[f"delta_h{h}", f"target_h{h}"]).copy()
    va = master[(master.year == 2020)].dropna(subset=[f"delta_h{h}", f"target_h{h}"]).copy()
    te = master[(master.year >= 2021)].dropna(subset=[f"delta_h{h}", f"target_h{h}"]).copy()

    # Leakage-safe station mean (fit on train only)
    mean_map = tr.groupby("tenTram")["doCaoDinhT"].mean().to_dict()
    gmean = float(tr["doCaoDinhT"].mean())

    for df in (tr, va, te):
        df["station_mean_peak"] = df["tenTram"].map(mean_map).fillna(gmean)
        df["y_t"] = df["doCaoDinhT"]

    station_maps[h] = mean_map

    Xtr = tr[FEAT].fillna(0)
    Xva = va[FEAT].fillna(0)
    Xte = te[FEAT].fillna(0)

    ytr_d = tr[f"delta_h{h}"].values
    yva_d = va[f"delta_h{h}"].values
    yte_d = te[f"delta_h{h}"].values

    ytr_abs = tr[f"target_h{h}"].values
    yva_abs = va[f"target_h{h}"].values
    yte_abs = te[f"target_h{h}"].values

    sw = make_weights(ytr_abs)
    mods = fit_blend(Xtr, ytr_d, sw, Xva, yva_d)

    pred_d_va = predict_blend(mods, Xva)
    pred_d_te = predict_blend(mods, Xte)

    pred_abs_va = va["y_t"].values + pred_d_va
    pred_abs_te = te["y_t"].values + pred_d_te

    # Conformal q90 from 2020 validation residuals
    val_resid = np.abs(yva_abs - pred_abs_va)
    q90 = float(np.quantile(val_resid, 0.90))

    met = {
        "horizon": h,
        "RMSE": rmse(yte_abs, pred_abs_te),
        "MAE": mae(yte_abs, pred_abs_te),
        "R2": r2(yte_abs, pred_abs_te),
        "conformal_q90": q90,
        "n_train": len(tr),
        "n_validation": len(va),
        "n_test": len(te),
    }
    metrics_peak.append(met)

    # Validation predictions
    va = va.copy()
    va["pred_delta"] = pred_d_va
    va["pred_peak"] = pred_abs_va
    va["pred_lo"] = pred_abs_va - q90
    va["pred_hi"] = pred_abs_va + q90
    va["event_day"] = va["ngay"] + pd.Timedelta(days=h)
    va["alarm_true"] = [peak_to_alarm(v) for v in yva_abs]
    va["alarm_pred"] = [peak_to_alarm(v) for v in pred_abs_va]
    va["flood_true"] = (va["alarm_true"] >= 1).astype(int)
    va["flood_pred"] = (va["alarm_pred"] >= 1).astype(int)
    preds_validation[h] = va

    # Test predictions
    te = te.copy()
    te["pred_delta"] = pred_d_te
    te["pred_peak"] = pred_abs_te
    te["pred_lo"] = pred_abs_te - q90
    te["pred_hi"] = pred_abs_te + q90
    te["event_day"] = te["ngay"] + pd.Timedelta(days=h)
    te["alarm_true"] = [peak_to_alarm(v) for v in yte_abs]
    te["alarm_pred"] = [peak_to_alarm(v) for v in pred_abs_te]
    te["flood_true"] = (te["alarm_true"] >= 1).astype(int)
    te["flood_pred"] = (te["alarm_pred"] >= 1).astype(int)
    preds_test[h] = te

    te[[
        "ngay", "event_day", "tenTram", f"target_h{h}", "y_t",
        "pred_delta", "pred_peak", "pred_lo", "pred_hi",
        "alarm_true", "alarm_pred", "flood_true", "flood_pred",
    ]].to_csv(OUT / f"pred_h{h}_test.csv", index=False)

    va[[
        "ngay", "event_day", "tenTram", f"target_h{h}", "y_t",
        "pred_delta", "pred_peak", "pred_lo", "pred_hi",
        "alarm_true", "alarm_pred", "flood_true", "flood_pred",
    ]].to_csv(OUT / f"pred_h{h}_validation.csv", index=False)

    models_h[h] = mods
    print(pd.Series(met).to_string())

pd.DataFrame(metrics_peak).to_csv(OUT / "metrics_water_level.csv", index=False)
print("\n=== TEST WATER-LEVEL METRICS (year ≥ 2021) ===")
print(pd.DataFrame(metrics_peak).to_string(index=False))


TRAIN / VALIDATION / TEST — t+1
horizon             1.000000
RMSE                0.065551
MAE                 0.047019
R2                  0.901581
conformal_q90       0.097867
n_train          8739.000000
n_validation      559.000000
n_test           1421.000000

=== TEST WATER-LEVEL METRICS (year ≥ 2021) ===
 horizon     RMSE      MAE       R2  conformal_q90  n_train  n_validation  n_test
       1 0.065551 0.047019 0.901581       0.097867     8739           559    1421


## 6. Binary flood warning (level ≥ I)

Binary positive class = administrative level ≥ I (≥ 1.40 m).  
Threshold is still calibrated on the external 2022 flood-event labels using Fβ (β=2)  
for reference, then we also report metrics under the consistent definition  
`flood = (alarm_level ≥ 1)`.

In [ ]:
flood_path = find_file("flood_binary_2022_5_stations_long.csv")
if flood_path is None:
    print("Flood labels not found; skipping external binary flood evaluation.")
    flood = pd.DataFrame(columns=["date", "station", "flood_binary"])
else:
    flood = pd.read_csv(flood_path)
flood["event_day"] = pd.to_datetime(flood["date"])
flood["tenTram"] = flood["station"].map(STATION_MAP)
flood = flood.rename(columns={"flood_binary": "flood_gt"})[["event_day", "tenTram", "flood_gt"]]

threshold_rows = []
flood_metric_rows = []
flood_eval_frames = []

for h in HORIZONS:
    test2022 = preds_test[h][preds_test[h]["event_day"].dt.year == 2022].copy()
    mrg = test2022.merge(flood, on=["event_day", "tenTram"], how="inner")

    if len(mrg) == 0:
        continue

    global_thr, global_fb = best_fbeta_threshold(mrg["flood_gt"], mrg["pred_peak"], BETA)

    station_thr = {}
    for st, g in mrg.groupby("tenTram"):
        if int(g["flood_gt"].sum()) >= 2:
            t, fb = best_fbeta_threshold(g["flood_gt"], g["pred_peak"], BETA)
            source = "station_2022_calibration"
        else:
            t, fb, source = global_thr, global_fb, "global_2022_fallback"

        station_thr[st] = t
        threshold_rows.append({
            "horizon": h,
            "tenTram": st,
            "threshold": float(t),
            "threshold_source": source,
            "station_positive_events_2022": int(g["flood_gt"].sum()),
            "beta": BETA,
        })

    mrg["flood_threshold"] = [station_thr.get(s, global_thr) for s in mrg["tenTram"]]
    mrg["flood_pred_ext"] = (mrg["pred_peak"] >= mrg["flood_threshold"]).astype(int)

    bm_ext = binary_metrics(mrg["flood_gt"], mrg["flood_pred_ext"], BETA)
    flood_metric_rows.append({
        "horizon": h,
        "method": "external_flood_labels_Fbeta",
        "global_threshold": float(global_thr),
        "global_fbeta": float(global_fb),
        "beta": BETA,
        **bm_ext,
    })

    bm_admin = binary_metrics(mrg["flood_true"], mrg["flood_pred"], BETA)
    flood_metric_rows.append({
        "horizon": h,
        "method": "admin_level_ge_I",
        "global_threshold": THR_I,
        "global_fbeta": None,
        "beta": BETA,
        **bm_admin,
    })

    mrg["horizon"] = h
    flood_eval_frames.append(mrg)

threshold_df = pd.DataFrame(threshold_rows)
flood_metrics_df = pd.DataFrame(flood_metric_rows)

threshold_df.to_csv(OUT / "flood_thresholds.csv", index=False)
flood_metrics_df.to_csv(OUT / "flood_metrics_binary.csv", index=False)

print("=== BINARY FLOOD METRICS ===")
print(flood_metrics_df.to_string(index=False))

print("\n=== THRESHOLD LOOKUP (from external labels) ===")
if not threshold_df.empty:
    print(threshold_df.pivot(index="tenTram", columns="horizon", values="threshold"))

if flood_eval_frames:
    pd.concat(flood_eval_frames, ignore_index=True).to_csv(
        OUT / "flood_event_eval_2022.csv", index=False
    )

=== BINARY FLOOD METRICS ===
 horizon                      method  global_threshold  global_fbeta  beta  precision   recall      f1    fbeta  support  pred_pos
       1 external_flood_labels_Fbeta          1.517114        0.6621   2.0   0.353659 0.852941 0.50000 0.665138       34        82
       1            admin_level_ge_I          1.400000           NaN   2.0   0.935323 0.817391 0.87239 0.838537      230       201

=== THRESHOLD LOOKUP (from external labels) ===
horizon         1
tenTram          
Nhà Bè   1.521799
Phú An   1.517114


In [8]:
mrg

,ngay,tenTram,doCaoDinhT,alarm_level,baoDongI,baoDongII,baoDongIII,rain,rain_max_intensity,rain_hours,...,pred_hi,event_day,alarm_true,alarm_pred,flood_true,flood_pred,flood_gt,flood_threshold,flood_pred_ext,horizon
0,2021-12-31,Nhà Bè,1.44,1,1,0,0,0.0,0.0,0,...,1.542422,2022-01-01,1,1,1,1,0,1.521799,0,1
1,2022-01-01,Nhà Bè,1.48,1,1,0,0,0.0,0.0,0,...,1.598226,2022-01-02,2,2,1,1,0,1.521799,0,1
2,2022-01-02,Nhà Bè,1.52,2,0,1,0,0.0,0.0,0,...,1.670219,2022-01-03,2,2,1,1,0,1.521799,1,1
3,2022-01-03,Nhà Bè,1.56,2,0,1,0,0.4,0.2,3,...,1.699843,2022-01-04,2,3,1,1,0,1.521799,1,1
4,2022-01-04,Nhà Bè,1.59,2,0,1,0,2.0,0.9,3,...,1.668354,2022-01-05,3,2,1,1,0,1.521799,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714,2022-12-26,Phú An,1.60,3,0,0,1,0.0,0.0,0,...,1.627978,2022-12-27,2,2,1,1,0,1.517114,1,1
715,2022-12-27,Phú An,1.52,2,0,1,0,0.0,0.0,0,...,1.520406,2022-12-28,1,1,1,1,0,1.517114,0,1
716,2022-12-28,Phú An,1.40,1,1,0,0,0.0,0.0,0,...,1.446787,2022-12-29,0,0,0,0,0,1.517114,0,1
717,2022-12-29,Phú An,1.39,0,0,0,0,0.0,0.0,0,...,1.454177,2022-12-30,1,0,1,0,0,1.517114,0,1


## 7. Severity classification (0 / I / II / III)

Evaluated on the full Test set (year ≥ 2021) using the merged 4-level scheme.

In [9]:
severity_rows = []

for h in HORIZONS:
    te = preds_test[h]
    true = te["alarm_true"].astype(int)
    pred = te["alarm_pred"].astype(int)

    report = classification_report(
        true, pred,
        labels=[0, 1, 2, 3],
        output_dict=True,
        zero_division=0,
    )

    for cls in ["0", "1", "2", "3"]:
        severity_rows.append({
            "horizon": h,
            "level": cls,
            "precision": report[cls]["precision"],
            "recall": report[cls]["recall"],
            "f1": report[cls]["f1-score"],
            "support": int(report[cls]["support"]),
        })

    print(f"\n=== TEST SEVERITY t+{h} (0 / I / II / III) ===")
    print(classification_report(
        true, pred,
        labels=[0, 1, 2, 3],
        target_names=["0", "I", "II", "III"],
        digits=3,
        zero_division=0,
    ))

severity_df = pd.DataFrame(severity_rows)
severity_df.to_csv(OUT / "severity_metrics_test.csv", index=False)


=== TEST SEVERITY t+1 (0 / I / II / III) ===
              precision    recall  f1-score   support

           0      0.928     0.975     0.951      1029
           I      0.634     0.537     0.581       216
          II      0.567     0.509     0.536       116
         III      0.792     0.700     0.743        60

    accuracy                          0.859      1421
   macro avg      0.730     0.680     0.703      1421
weighted avg      0.848     0.859     0.852      1421



## 8. SHAP interpretability (optional)

In [10]:
try:
    import shap
    H_SHAP = 1
    model = models_h[H_SHAP]["lgb"]
    te = preds_test[H_SHAP]
    X_shap = te.nlargest(min(100, len(te)), "pred_peak")[FEAT].fillna(0)
    sv = shap.TreeExplainer(model).shap_values(X_shap)
    imp = pd.Series(np.abs(sv).mean(axis=0), index=FEAT).sort_values(ascending=False)
    imp.head(20).to_csv(OUT / "shap_top_h1_test.csv")
    print(imp.head(20).to_string())
except Exception as e:
    print("SHAP skipped:", repr(e))

y_t                0.070380
tide_max           0.047537
spring_neap_sin    0.022747
tide_mean          0.013046
spring_neap_cos    0.012524
tide_range_lag5    0.012374
tide_mean_lag1     0.009689
tide_max_lag1      0.006810
tide_max_lag5      0.006748
tide_max_lag3      0.006635
tide_range_lag7    0.004605
doy                0.004031
tide_max_lag2      0.003523
tide_range_lag3    0.002900
doy_cos            0.002562
month_cos          0.002477
doy_sin            0.002391
tide_range_lag2    0.002333
tide_min           0.002060
tide_range         0.002015


## 9. Export deployment package

Package contents:
- Ensemble models for t+1
- `feature_schema.json`
- `thresholds.json`
- `conformal_q90.json`
- `station_mean_maps.json`
- Metrics & prediction files
- `predict_saved.py` (inference helper)
- `metadata.json`

In [11]:
# ---- Schema & configuration ----
with open(MODEL_DIR / "feature_schema.json", "w", encoding="utf-8") as f:
    json.dump({"features": FEAT, "horizons": HORIZONS}, f, ensure_ascii=False, indent=2)

with open(MODEL_DIR / "thresholds.json", "w", encoding="utf-8") as f:
    json.dump(threshold_df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

with open(MODEL_DIR / "station_mean_maps.json", "w", encoding="utf-8") as f:
    json.dump(
        {str(k): {str(a): float(b) for a, b in v.items()} for k, v in station_maps.items()},
        f, ensure_ascii=False, indent=2,
    )

with open(MODEL_DIR / "conformal_q90.json", "w", encoding="utf-8") as f:
    json.dump(
        {str(h): float(m["conformal_q90"]) for h, m in zip(HORIZONS, metrics_peak)},
        f, indent=2,
    )

# ---- Models ----
for h, mods in models_h.items():
    hdir = MODEL_DIR / f"h{h}"
    hdir.mkdir(exist_ok=True)
    if "lgb" in mods:
        joblib.dump(mods["lgb"], hdir / "lgb.joblib")
    if "xgb" in mods:
        mods["xgb"].save_model(str(hdir / "xgb.json"))
    if "cat" in mods:
        mods["cat"].save_model(str(hdir / "cat.cbm"))

# ---- Metrics ----
pd.DataFrame(metrics_peak).to_csv(MODEL_DIR / "metrics_water_level.csv", index=False)
flood_metrics_df.to_csv(MODEL_DIR / "metrics_flood_binary.csv", index=False)
severity_df.to_csv(MODEL_DIR / "metrics_severity_test.csv", index=False)

# ---- Metadata ----
metadata = {
    "model_version": "water_flood_t1",
    "training_split": "train ≤ 2019; validation = 2020; test ≥ 2021",
    "target": "delta peak water level; final = y_t + predicted_delta",
    "prediction_horizons": [1],
    "ensemble": ["lightgbm", "xgboost", "catboost"],
    "blend": "equal_weight",
    "severity_levels": {
        "0": "< 1.40 m",
        "I": "1.40 – < 1.50 m",
        "II": "1.50 – < 1.60 m",
        "III": "≥ 1.60 m (includes former IV)",
    },
    "binary_flood_definition": "alarm_level ≥ 1 (i.e. ≥ 1.40 m)",
    "conformal": "q90 calibrated from 2020 validation residuals",
    "stations": stations,
}
with open(MODEL_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# ---- Inference helper ----
inference_code = '''import json
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb

try:
    from catboost import CatBoostRegressor
except Exception:
    CatBoostRegressor = None

PACKAGE_DIR = Path(__file__).resolve().parent

with open(PACKAGE_DIR / "feature_schema.json", encoding="utf-8") as f:
    SCHEMA = json.load(f)
with open(PACKAGE_DIR / "thresholds.json", encoding="utf-8") as f:
    THRESHOLDS = json.load(f)
with open(PACKAGE_DIR / "station_mean_maps.json", encoding="utf-8") as f:
    STATION_MEANS = json.load(f)
with open(PACKAGE_DIR / "conformal_q90.json", encoding="utf-8") as f:
    Q90 = json.load(f)

FEAT = SCHEMA["features"]
THR_I, THR_II, THR_III = 1.40, 1.50, 1.60


def load_models(h: int):
    d = PACKAGE_DIR / f"h{h}"
    models = []
    if (d / "lgb.joblib").exists():
        models.append(joblib.load(d / "lgb.joblib"))
    if (d / "xgb.json").exists():
        m = xgb.XGBRegressor()
        m.load_model(str(d / "xgb.json"))
        models.append(m)
    if (d / "cat.cbm").exists() and CatBoostRegressor is not None:
        m = CatBoostRegressor()
        m.load_model(str(d / "cat.cbm"))
        models.append(m)
    return models


def station_threshold(station: str, h: int) -> float:
    vals = [
        r["threshold"] for r in THRESHOLDS
        if int(r["horizon"]) == int(h) and r["tenTram"] == station
    ]
    if vals:
        return float(vals[0])
    same_h = [r["threshold"] for r in THRESHOLDS if int(r["horizon"]) == int(h)]
    return float(np.median(same_h)) if same_h else THR_I


def predict_saved(df: pd.DataFrame, h: int = 1) -> pd.DataFrame:
    x = df.copy()
    x["y_t"] = x["doCaoDinhT"]

    if "station_mean_peak" not in x.columns:
        means = STATION_MEANS.get(str(h), {})
        gm = np.mean(list(means.values())) if means else float(x["y_t"].mean())
        x["station_mean_peak"] = [means.get(s, gm) for s in x["tenTram"]]

    X = x[FEAT].fillna(0)
    pred_delta = np.mean([m.predict(X) for m in load_models(h)], axis=0)
    pred_peak = x["y_t"].to_numpy() + pred_delta
    q = float(Q90[str(h)])

    out = x[["tenTram"]].copy()
    out["pred_delta"] = pred_delta
    out["pred_peak"] = pred_peak
    out["pred_lo"] = pred_peak - q
    out["pred_hi"] = pred_peak + q
    out["flood_threshold"] = [station_threshold(s, h) for s in out["tenTram"]]
    out["flood_warning"] = (out["pred_peak"] >= out["flood_threshold"]).astype(int)
    out["standby_warning"] = (out["pred_hi"] >= out["flood_threshold"]).astype(int)

    # 4-level severity (IV merged into III)
    out["alarm_level"] = np.select(
        [
            out["pred_peak"] >= THR_III,
            out["pred_peak"] >= THR_II,
            out["pred_peak"] >= THR_I,
        ],
        [3, 2, 1],
        default=0,
    )
    return out
'''

(MODEL_DIR / "predict_saved.py").write_text(inference_code, encoding="utf-8")

zip_path = shutil.make_archive(str(OUT / "water_flood_t1"), "zip", root_dir=MODEL_DIR)
print("PACKAGE :", zip_path)
print("MODEL DIR:", MODEL_DIR)

PACKAGE : /kaggle/working/water_flood_t1.zip
MODEL DIR: /kaggle/working/water_flood_t1


## 10. Output summary

| File | Description |
|------|-------------|
| `metrics_water_level.csv` | RMSE / MAE / R² on Test (≥ 2021) |
| `flood_metrics_binary.csv` | Binary flood metrics (external labels + admin ≥ I) |
| `severity_metrics_test.csv` | 0 / I / II / III classification on Test |
| `flood_thresholds.csv` | Per-station thresholds calibrated on 2022 |
| `pred_h1_test.csv` | Test predictions |
| `pred_h1_validation.csv` | Validation predictions (conformal calibration) |
| **`water_flood_t1.zip`** | Full deployment package |